In [149]:
import pandas as pd
import numpy as np
from PIL import Image

df = pd.read_csv("Reset_Start_Screen_Idle_100Mhz_Better_Resolution(gusmanb pi pico firmware).csv")

In [150]:
df = df.iloc[2800:].copy() #drops the first broken scanline
df

,CS,RS,WR,D00,D01,D02,D03,D04,D05,D06,D07,D08,D09,D10,D11,D12,D13,D14,D15
2800,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
2801,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
2802,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
2803,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
2804,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32251,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
32252,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
32253,0,1,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0
32254,0,1,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0


In [158]:
def getRisingEdges(df):
    rising_edge = False
    falling_edge = False
    df_16bit = pd.DataFrame()
    lastWR = None
    for index, row in df.iterrows():
        if (lastWR != row['WR']):
            rising_edge = False
            falling_edge = False
            if (row['WR'] == 1):
                rising_edge = True
                new_row = row.drop(labels=['CS','RS','WR'])
                df_new_row = pd.DataFrame([new_row])
                df_16bit = pd.concat([df_16bit, df_new_row])
            else:
                falling_edge = True
        lastWR = row['WR']
    df_16bit = df_16bit.reset_index(drop=True)
    return(df_16bit)

def BitsToColor(bits):
    R_str = str(bits['D11'])+str(bits['D12'])+str(bits['D13'])+str(bits['D14'])+str(bits['D15'])
    G_str = str(bits['D05'])+str(bits['D06'])+str(bits['D07'])+str(bits['D08'])+str(bits['D09'])+str(bits['D10'])
    B_str = str(bits['D00'])+str(bits['D01'])+str(bits['D02'])+str(bits['D03'])+str(bits['D04'])
    R = int((int(R_str, 2)/31)*255)
    G = int((int(G_str, 2)/63)*255)
    B = int((int(B_str, 2)/31)*255)
    return((R,G,B))
    
def convertToColorList(df):
    output_list = []
    for index, row in df.iterrows():
        output_list.append(BitsToColor(row))
    return(output_list)

In [159]:
df_16bit = getRisingEdges(df)


In [160]:
pixelList = convertToColorList(df_16bit)

In [161]:
pixelarray = np.array(pixelList, dtype=np.uint8)
rows = len(pixelarray)//320
imagearray = np.resize(pixelarray,(rows,320,3))
# Use PIL to create an image from the new array of pixels
pil_im = Image.fromarray(imagearray)
pil_im

In [157]:
pil_im.save("Decoded_Start_Screen.png")